# 03 — Separate model results and comparison

Run any of the four 02x notebooks first. Only finished models will appear. Figures saved to results_v2/comparison/.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc
from IPython.display import display
ROOT=Path('results_v2');OUT=ROOT/'comparison';OUT.mkdir(parents=True,exist_ok=True)
MODEL_NAMES={'resnet50':'ResNet50','densenet121':'DenseNet121','efficientnet_b7':'EfficientNet-B7','vit_base_patch16_224':'ViT-B/16'}
LABELS=['bdb','cb','db','srjd'];NAMES=['Blue','Compound','Dermal','Spitz/Reed']
models=[m for m in MODEL_NAMES if (ROOT/m/'metrics.json').exists()]
assert models,'No completed models yet. Run at least one 02x notebook.'
metrics=pd.DataFrame([json.loads((ROOT/m/'metrics.json').read_text(encoding='utf-8')) for m in models])
display(metrics[['model','accuracy','balanced_accuracy','macro_f1','macro_roc_auc_ovr','training_seconds']].round(3))

In [ ]:
# Four key metrics, all compared only where models have completed.
fields=['accuracy','balanced_accuracy','macro_f1','macro_roc_auc_ovr']
chart=metrics.set_index('model')[fields].rename(index=MODEL_NAMES)
ax=chart.plot.bar(figsize=(10,5),rot=0);ax.set(ylim=(0,1),ylabel='Score',title='CNN vs ViT — held-out test set')
plt.tight_layout();plt.savefig(OUT/'model_comparison.png',dpi=300);plt.show()

In [ ]:
# One confusion matrix and ROC for each available model.
for m in models:
    p=pd.read_csv(ROOT/m/'predictions.csv')
    y=p.true_index.to_numpy();pred=p.pred_index.to_numpy()
    cm=confusion_matrix(y,pred,labels=range(4))
    fig,ax=plt.subplots(figsize=(6,5));im=ax.imshow(cm,cmap='Blues');fig.colorbar(im,ax=ax)
    ax.set_xticks(range(4),NAMES,rotation=35,ha='right');ax.set_yticks(range(4),NAMES)
    ax.set(xlabel='Predicted',ylabel='Actual',title=f'{MODEL_NAMES[m]} — confusion matrix')
    for i in range(4):
        for j in range(4):ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
    plt.tight_layout();plt.savefig(OUT/f'{m}_confusion.png',dpi=300);plt.show()
    fig,ax=plt.subplots(figsize=(6,5))
    for k,lbl in enumerate(LABELS):
        fpr,tpr,_=roc_curve((y==k).astype(int),p[f'prob_{lbl}'].to_numpy())
        ax.plot(fpr,tpr,label=f'{NAMES[k]} (AUC={auc(fpr,tpr):.2f})')
    ax.plot([0,1],[0,1],'k--',linewidth=1);ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'{MODEL_NAMES[m]} — ROC OvR')
    ax.legend();plt.tight_layout();plt.savefig(OUT/f'{m}_roc.png',dpi=300);plt.show()

In [ ]:
# Comparison of saved training histories (validation macro F1, if available)
fig,ax=plt.subplots(figsize=(8,5))
for m in models:
    h=pd.read_csv(ROOT/m/'history.csv')
    ax.plot(h.epoch,h.val_macro_f1,marker='o',label=MODEL_NAMES[m])
ax.set(xlabel='Epoch',ylabel='Validation macro F1',title='Validation performance',ylim=(0,1));ax.legend()
plt.tight_layout();plt.savefig(OUT/'validation_macro_f1.png',dpi=300);plt.show()
metrics.to_csv(OUT/'comparison_metrics.csv',index=False)